In [13]:
import pandas as pd

# Завантажуємо основний датасет
# low_memory=False допоможе уникнути попереджень про різні типи даних у колонках
df = pd.read_csv('survey_results_public.csv', low_memory=False)

# Завантажуємо схему без вказання індексу, щоб точно не було помилок
schema_df = pd.read_csv('survey_results_schema.csv')

# Налаштовуємо Pandas для зручного перегляду всіх колонок
pd.set_option('display.max_columns', 85)

print("Дані успішно завантажено!")

# Виведемо назви колонок схеми, щоб перевірити їх
print("Колонки у файлі схеми:", schema_df.columns.tolist())

Дані успішно завантажено!
Колонки у файлі схеми: ['qid', 'qname', 'question', 'type', 'sub', 'sq_id']


In [14]:
# Налаштовуємо індекс для зручної роботи зі схемою питань
schema_df.set_index('qname', inplace=True)

# Визначаємо кількість респондентів через атрибут .shape (індекс [0] — це рядки)
total_respondents = df.shape[0]

# Виводимо результат
print(f"Загальна кількість респондентів: {total_respondents}")

# Перевіримо, чи працює наша схема (дізнаємося, про що питання 'MainBranch')
print("\nОпис колонки MainBranch:")
print(schema_df.loc['MainBranch', 'question'])

Загальна кількість респондентів: 49191

Опис колонки MainBranch:
Are you someone who writes code? Please select one of the following options that best describes you today.


In [15]:
# Отримуємо список питань з файлу схеми (поле qname)
# Оскільки ми вже зробили qname індексом раніше, просто беремо індекс
questions_from_schema = set(schema_df.index)

# Отримуємо список усіх колонок, які є в нашому основному датасеті df
columns_in_df = set(df.columns)

# Використовуємо інтерсекцію (перетин) множин, щоб знайти колонки, 
# які є і в схемі, і в датасеті (фільтрація лише реальних питань)
relevant_columns = list(questions_from_schema.intersection(columns_in_df))

# Фільтруємо наш основний датасет, залишаючи тільки ці колонки
df_questions_only = df[relevant_columns]

# Підраховуємо респондентів, у яких немає ЖОДНОГО пропущеного значення (NaN) 
# у вибраних колонках. Метод dropna() видаляє рядки з пропусками.
respondents_all_answered = df_questions_only.dropna().shape[0]

print(f"Кількість питань для перевірки: {len(relevant_columns)}")
print(f"Кількість респондентів, які відповіли на ВСІ запитання: {respondents_all_answered}")

Кількість питань для перевірки: 126
Кількість респондентів, які відповіли на ВСІ запитання: 0


In [16]:
# Обчислюємо середнє значення (Mean)
mean_exp = df['WorkExp'].mean()

# Обчислюємо медіану (Median)
median_exp = df['WorkExp'].median()

# Обчислюємо моду (Mode)
# Використовуємо [0], бо метод .mode() повертає Series
mode_exp = df['WorkExp'].mode()[0]

# Виводимо результати у зручному форматі
print("Міри центральної тенденції для досвіду роботи (WorkExp):")
print(f"1. Середнє значення: {mean_exp:.2f} років")
print(f"2. Медіана:          {median_exp:.2f} років")
print(f"3. Мода:             {mode_exp:.2f} років")

Міри центральної тенденції для досвіду роботи (WorkExp):
1. Середнє значення: 13.37 років
2. Медіана:          10.00 років
3. Мода:             10.00 років


In [17]:
# Спершу перевіримо через схему, чи правильно ми обрали колонку
print("Опис питання про формат роботи:")
print(schema_df.loc['RemoteWork', 'question'])

# Подивимося на всі варіанти відповідей, які є в цій колонці
print("\nВаріанти відповідей у колонці RemoteWork:")
print(df['RemoteWork'].value_counts())

# Фільтруємо респондентів, які обрали саме 'Remote'
# (Зверни увагу на точну назву варіанту у виводі вище, зазвичай це 'Remote')
remote_workers_count = df[df['RemoteWork'] == 'Remote'].shape[0]

print(f"\nКількість респондентів, які працюють ВІДДАЛЕНО: {remote_workers_count}")

Опис питання про формат роботи:
Which best describes your current work situation?

Варіанти відповідей у колонці RemoteWork:
RemoteWork
Remote                                                                          10931
Hybrid (some remote, leans heavy to in-person)                                   6732
In-person                                                                        6042
Hybrid (some in-person, leans heavy to flexibility)                              5831
Your choice (very flexible, you can come in when you want or just as needed)     4244
Name: count, dtype: int64

Кількість респондентів, які працюють ВІДДАЛЕНО: 10931


In [18]:
# Рахуємо загальну кількість респондентів, які відповіли на питання про мови
total_with_languages = df['LanguageHaveWorkedWith'].dropna().shape[0]

#  Шукаємо респондентів, у яких в списку мов є 'Python'
# Метод .str.contains допомагає знайти підрядок у великому рядку з багатьма мовами
python_users_count = df['LanguageHaveWorkedWith'].str.contains('Python', na=False).sum()

# Розраховуємо відсоток
python_percentage = (python_users_count / total_with_languages) * 100

# Виводимо результат у відсотковому форматі
print(f"Кількість респондентів, що використовують Python: {python_users_count}")
print(f"Відсоток респондентів, які програмують на Python: {python_percentage:.1f}%")

Кількість респондентів, що використовують Python: 18466
Відсоток респондентів, які програмують на Python: 58.3%


In [19]:
# Спершу перевіримо через схему опис колонки LearnCode
print("Опис питання про способи навчання:")
print(schema_df.loc['LearnCode', 'question'])

# Шукаємо респондентів, які обрали 'Online Courses'
# Використовуємо .str.contains, бо способів навчання зазвичай кілька
online_learners_count = df['LearnCode'].str.contains('Online Courses', na=False).sum()

print(f"\nКількість респондентів, які навчалися через онлайн-курси: {online_learners_count}")

Опис питання про способи навчання:
How did you learn to code in the past year? Select all that apply.

Кількість респондентів, які навчалися через онлайн-курси: 10973


In [20]:
# Фільтруємо датасет: залишаємо тільки тих, хто працює з Python
python_devs = df[df['LanguageHaveWorkedWith'].str.contains('Python', na=False)]

# Групуємо за країною та розраховуємо середнє (mean) і медіану (median) для зарплати
# Метод .agg дозволяє обчислити кілька показників одночасно
geo_compensation = python_devs.groupby('Country')['ConvertedCompYearly'].agg(['mean', 'median'])

# Перейменовуємо колонки для кращого вигляду (згідно з фінальним результатом)
geo_compensation.columns = ['Середня компенсація', 'Медіанна компенсація']

# Сортуємо за медіаною (від найбільшої), щоб побачити топ країн
geo_compensation = geo_compensation.sort_values(by='Медіанна компенсація', ascending=False)

# Виводимо перші 15 країн у табличному форматі
print("Географічний аналіз компенсації Python-розробників:")
geo_compensation.head(15)

Географічний аналіз компенсації Python-розробників:


,Середня компенсація,Медіанна компенсація
Country,,
Oman,390135.000000,390135.0
Andorra,226103.500000,226103.5
United States of America,173298.590211,150000.0
Israel,135828.365385,142594.0
Switzerland,156456.600000,142592.0
Nomadic,120131.571429,139218.0
Ireland,120523.918919,116015.0
Luxembourg,116014.714286,109054.0
Kyrgyzstan,106008.500000,106008.5


In [21]:
# Сортуємо весь датасет за колонкою компенсації у спадному порядку
# .sort_values допоможе поставити найбільші зарплати на початок
top_paid_respondents = df.sort_values(by='ConvertedCompYearly', ascending=False)

# Вибираємо топ-5 респондентів та беремо лише колонку з освітою (EdLevel)
# Використовуємо .head(5), щоб отримати рівно 5 записів
top_5_education = top_paid_respondents[['EdLevel', 'ConvertedCompYearly']].head(5)

# Виводимо результат
print("Рівні освіти 5 респондентів з найбільшою компенсацією:")
print(top_5_education)

Рівні освіти 5 респондентів з найбільшою компенсацією:
                                               EdLevel  ConvertedCompYearly
34267              Associate degree (A.A., A.S., etc.)           50000000.0
28700  Master’s degree (M.A., M.S., M.Eng., MBA, etc.)           33552715.0
43143              Associate degree (A.A., A.S., etc.)           18387548.0
35353     Bachelor’s degree (B.A., B.S., B.Eng., etc.)           15430267.0
45971  Master’s degree (M.A., M.S., M.Eng., MBA, etc.)           13921760.0


In [22]:
# Створюємо колонку-маркер: True, якщо людина знає Python, False — якщо ні
df['IsPythonUser'] = df['LanguageHaveWorkedWith'].str.contains('Python', na=False)

# Групуємо за віком та рахуємо середнє значення маркера IsPythonUser
# Оскільки True = 1, а False = 0, середнє значення якраз дасть нам частку (відсоток)
python_by_age = df.groupby('Age')['IsPythonUser'].mean() * 100

# Перетворюємо результат у гарний DataFrame
python_by_age_df = python_by_age.reset_index()
python_by_age_df.columns = ['Вікова категорія', 'Відсоток Python-користувачів']

# Виводимо результат
print("Популярність Python за віковими групами:")
python_by_age_df

Популярність Python за віковими групами:


,Вікова категорія,Відсоток Python-користувачів
0,18-24 years old,40.000000
1,25-34 years old,36.939282
2,35-44 years old,36.719281
3,45-54 years old,38.629482
4,55-64 years old,37.242955
5,65 years or older,31.634820
6,Prefer not to say,31.216931


In [23]:
# Розраховуємо 75-й перцентиль компенсації
# Це значення, вище якого заробляють лише 25% найбагатших респондентів
comp_75_percentile = df['ConvertedCompYearly'].quantile(0.75)

# Фільтруємо респондентів за двома критеріями:
# - зарплата вище 75 перцентиля
# - формат роботи "Remote"
high_paid_remote = df[
    (df['ConvertedCompYearly'] > comp_75_percentile) & 
    (df['RemoteWork'] == 'Remote')
]

# Визначаємо найрозповсюдженіші індустрії в цій групі
# Видаляємо пропуски та підраховуємо частоту
top_industries = high_paid_remote['Industry'].dropna().value_counts()

# Виводимо результат
print(f"75-й перцентиль зарплати: ${comp_75_percentile:,.2f}")
print(f"Кількість відібраних респондентів: {high_paid_remote.shape[0]}")
print("\nТоп індустрій серед високооплачуваних віддалених працівників:")
print(top_industries.head(10))

75-й перцентиль зарплати: $120,596.00
Кількість відібраних респондентів: 2461

Топ індустрій серед високооплачуваних віддалених працівників:
Industry
Software Development                          1186
Fintech                                        190
Healthcare                                     188
Other:                                         176
Internet, Telecomm or Information Services     138
Banking/Financial Services                      88
Government                                      78
Media & Advertising Services                    75
Retail and Consumer Services                    65
Transportation, or Supply Chain                 63
Name: count, dtype: int64
